In [155]:
import json
from tqdm import tqdm
import re
import random
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
from utils import *
import os

In [2]:
# Set a random seed
random_seed = 42
random.seed(random_seed)

# Set a random seed for PyTorch (for GPU as well)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

In [5]:
# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

In [110]:
def get_bert_embeddings_batch(texts, batch_size=32):
    """
    Get BERT embeddings for a batch of texts with proper mean pooling using attention masks
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize and encode batch of texts
        encoding = tokenizer.batch_encode_plus(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors='pt',
            add_special_tokens=True,
        )
        
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        
        # Generate embeddings using BERT model
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
            word_embeddings = outputs.last_hidden_state  # Shape: (batch_size, seq_len, hidden_size)
            
            # Mean pooling with attention mask
            # Expand attention mask to match embedding dimensions
            attention_mask_expanded = attention_mask.unsqueeze(-1).expand(word_embeddings.size()).float()
            
            # Apply attention mask to embeddings
            masked_embeddings = word_embeddings * attention_mask_expanded
            
            # Sum embeddings and divide by actual token count (excluding padding)
            sum_embeddings = torch.sum(masked_embeddings, dim=1)  # Shape: (batch_size, hidden_size)
            token_counts = torch.clamp(attention_mask.sum(dim=1, keepdim=True).float(), min=1e-9)
            
            # Mean pooled embeddings
            batch_embeddings = sum_embeddings / token_counts  # Shape: (batch_size, hidden_size)
            
        all_embeddings.append(batch_embeddings)
    
    return torch.cat(all_embeddings, dim=0)

In [111]:
qrel_id_to_query_text = {}
with open('data/test_queries.jsonl', 'r') as f:
	for line in f:
		item = json.loads(line)
		qrel_id_to_query_text[item['query_id']] = item['question']

In [112]:
all_query_texts = list(qrel_id_to_query_text.values())
query_embds = get_bert_embeddings_batch(all_query_texts, batch_size=16)

100%|██████████| 138/138 [01:02<00:00,  2.22it/s]


In [115]:
torch.save(query_embds, 'data/bert_query_embeddings.pt')

In [116]:
qrel_id_to_doc_text = {}
with open('data/test_documents.jsonl', 'r') as f:
	for line in f:
		item = json.loads(line)
		qrel_id_to_doc_text[item['doc_id']] = item['text']

In [119]:
all_doc_texts = list(qrel_id_to_doc_text.values())
doc_embds = get_bert_embeddings_batch(all_doc_texts, batch_size=1)

100%|██████████| 20968/20968 [37:54<00:00,  9.22it/s]


In [121]:
torch.save(doc_embds, 'data/bert_doc_embeddings.pt')

In [139]:
all_sims = (query_embds @ doc_embds.T)
all_sims.shape

torch.Size([2195, 20968])

In [141]:
with open("data/qrel_dict.json", "r") as f:
    qrel_dict = json.load(f)

In [152]:
doc_ids = list(qrel_id_to_doc_text.keys())

In [153]:
dcgs = []
mrrs = []
recalls = []

In [154]:
for i, (qrel_id, _) in tqdm(enumerate(qrel_id_to_query_text.items()), total=len(qrel_id_to_query_text)):
	similarities = all_sims[i]
	
	# rank and evaluate
	ranked_doc_tensor_indices = torch.argsort(similarities, descending=True)
	ranked_doc_ids = [
		doc_ids[int(idx)] for idx in ranked_doc_tensor_indices
	]

	good_doc_ids = qrel_dict[str(qrel_id)]

	dgc = DCG_at_k(good_doc_ids, ranked_doc_ids[:10])
	mrr = MRR_at_k(good_doc_ids, ranked_doc_ids[:10])
	recall = recall_at_k(good_doc_ids, ranked_doc_ids[:100])
	dcgs.append(dgc)
	mrrs.append(mrr)
	recalls.append(recall)

100%|██████████| 2195/2195 [00:57<00:00, 38.17it/s]


In [156]:
os.makedirs("results/task1", exist_ok=True)
with open("results/task1/bert_results.txt", "w") as f:
	f.write(
		f"Average DCG@10: {torch.mean(torch.tensor(dcgs))/torch.max(torch.tensor(dcgs)):.4f}\n"
	)
	f.write(f"Average MRR@10: {torch.mean(torch.tensor(mrrs)):.4f}\n")
	f.write(f"Average Recall@100: {torch.mean(torch.tensor(recalls)):.4f}\n")